# Artificial Intelligence — Exercise 1
## Football Analysis: International Football Results (1872–2024)

**Dataset:** [Kaggle — International Football Results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)

---

## Step 1: Load the CSV

In [ ]:
import pandas as pd

df = pd.read_csv("results.csv")
df.head()

---
## Section 1: Basic Exploration

We explore the fundamental structure of the dataset: size, date range, number of unique countries, and most frequent home teams.

### Question 1: How many matches are in the dataset?

`df.shape` returns a tuple of `(rows, columns)`. The number of rows equals the number of matches.

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"Number of matches: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

> **Answer:** There are **2,776 matches** in the dataset.

### Question 2: What is the earliest and latest year in the data?

We parse the `date` column as datetime and use `.min()` / `.max()` to find the time range.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

print(f"Earliest match: {df['date'].min().date()}")
print(f"Latest match:   {df['date'].max().date()}")

> **Answer:** The dataset spans from **30 November 1872** to **30 June 2024** — over 150 years of international football.

### Question 3: How many unique countries are there?

We combine `home_team` and `away_team` using `pd.concat`, then call `.nunique()` to count distinct nations.

In [ ]:
all_teams = pd.concat([df["home_team"], df["away_team"]])
print(f"Unique countries: {all_teams.nunique()}")

> **Answer:** There are **94 unique countries** in the dataset.

### Question 4: Which team appears most frequently as home team?

`.value_counts()` on `home_team` ranks teams by the number of times they hosted a match.

In [ ]:
print("Top 5 home teams:")
print(df["home_team"].value_counts().head())

> **Answer:** **Australia and Costa Rica** both appear most frequently as home team (40 times each).

---
## Section 2: Goals Analysis

We create a `total_goals` column and use it to explore scoring patterns across the dataset.

In [ ]:
# Create total goals column
df["total_goals"] = df["home_score"] + df["away_score"]
df[["home_team", "away_team", "home_score", "away_score", "total_goals"]].head()

### Question 1: What is the average number of goals per match?

In [ ]:
avg_goals = df["total_goals"].mean()
print(f"Average goals per match: {avg_goals:.2f}")

> **Answer:** The average number of goals per match is **2.32**.

### Question 2: What is the highest scoring match?

We use `.idxmax()` to locate the row with the highest `total_goals`.

In [ ]:
idx = df["total_goals"].idxmax()
highest = df.loc[idx]
print(f"Match: {highest['home_team']} {int(highest['home_score'])} - {int(highest['away_score'])} {highest['away_team']}")
print(f"Date:  {highest['date'].date()}")
print(f"Total goals: {int(highest['total_goals'])}")

> **Answer:** The highest scoring match was **Czech Republic 4 – 4 Russia** (22 July 1905), with **8 total goals**.

### Question 3: Are more goals scored at home or away?

In [ ]:
home_total = df["home_score"].sum()
away_total = df["away_score"].sum()

print(f"Total home goals: {home_total}")
print(f"Total away goals: {away_total}")
print(f"More goals scored: {'at home' if home_total > away_total else 'away'}")

> **Answer:** More goals are scored **at home** (3,591) than away (2,855). This is consistent with the well-known home advantage effect in football.

### Question 4: What is the most common total goals value?

In [ ]:
most_common = df["total_goals"].mode()[0]
count = (df["total_goals"] == most_common).sum()
print(f"Most common total goals: {most_common} (occurs in {count} matches)")

> **Answer:** The most common total goals value is **2** — meaning many matches end 1–1 or 2–0.

---
## Section 3: Match Results

We classify each match as a Home Win, Away Win, or Draw using a custom function applied with `.apply()`.

In [ ]:
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"

df["result"] = df.apply(match_result, axis=1)
print(df["result"].value_counts())

### Question 1: What percentage of matches are home wins?

In [ ]:
result_counts = df["result"].value_counts()
total = len(df)

for outcome in ["Home Win", "Away Win", "Draw"]:
    count = result_counts.get(outcome, 0)
    pct = count / total * 100
    print(f"{outcome}: {count} ({pct:.1f}%)")

> **Answer:** **42.9% of matches are home wins** — compared to 28.2% away wins and 28.9% draws.

### Question 2: Does home advantage exist?

In [ ]:
home_win_pct = result_counts["Home Win"] / total * 100
away_win_pct = result_counts["Away Win"] / total * 100
gap = home_win_pct - away_win_pct

print(f"Home win rate: {home_win_pct:.1f}%")
print(f"Away win rate: {away_win_pct:.1f}%")
print(f"Gap: {gap:.1f} percentage points")
print(f"Home advantage exists: {home_win_pct > away_win_pct}")

> **Answer:** Yes — home teams win **14.7 percentage points more often** than away teams. Home advantage is clearly confirmed in this dataset.

### Question 3: Which country has the most wins historically?

We sum home wins and away wins per country to get a true all-time wins total.

In [ ]:
home_wins = df[df["result"] == "Home Win"].groupby("home_team").size()
away_wins = df[df["result"] == "Away Win"].groupby("away_team").size()

total_wins = home_wins.add(away_wins, fill_value=0).astype(int)
total_wins = total_wins.sort_values(ascending=False)

print("Top 10 countries by total wins:")
print(total_wins.head(10))

> **Answer:** **Namibia** leads with 33 total wins in this dataset.

---
## Section 4: Visualization

We produce three charts:
1. Histogram of goals per match
2. Bar chart of match outcomes
3. Top 10 teams by total wins

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

# --- Chart 1: Histogram of goals per match ---
ax1 = axes[0]
ax1.set_facecolor('#161b22')
ax1.hist(
    df["total_goals"],
    bins=range(0, df["total_goals"].max() + 2),
    color='#58a6ff', edgecolor='#0d1117', alpha=0.9, rwidth=0.85
)
ax1.axvline(df["total_goals"].mean(), color='#ffa657', linestyle='--',
            linewidth=2, label=f'Avg: {df["total_goals"].mean():.1f}')
ax1.set_title("Distribution of Goals Per Match", color='white', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel("Total Goals", color='#8b949e', fontsize=11)
ax1.set_ylabel("Number of Matches", color='#8b949e', fontsize=11)
ax1.tick_params(colors='#8b949e')
for spine in ['top', 'right']:
    ax1.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax1.spines[spine].set_color('#30363d')
ax1.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='white')

# --- Chart 2: Bar chart of match outcomes ---
ax2 = axes[1]
ax2.set_facecolor('#161b22')
outcome_labels = ["Home Win", "Away Win", "Draw"]
outcome_values = [result_counts.get(o, 0) for o in outcome_labels]
outcome_colors = ['#3fb950', '#f78166', '#d2a8ff']
bars = ax2.bar(outcome_labels, outcome_values, color=outcome_colors, edgecolor='#0d1117', width=0.6)
for bar, val in zip(bars, outcome_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f'{val}\n({val / total * 100:.1f}%)',
             ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax2.set_title("Match Outcomes", color='white', fontsize=13, fontweight='bold', pad=12)
ax2.set_ylabel("Number of Matches", color='#8b949e', fontsize=11)
ax2.tick_params(colors='#8b949e')
for spine in ['top', 'right']:
    ax2.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax2.spines[spine].set_color('#30363d')

# --- Chart 3: Top 10 teams by total wins ---
ax3 = axes[2]
ax3.set_facecolor('#161b22')
top10 = total_wins.head(10)
colors = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657',
          '#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657']
bars3 = ax3.barh(top10.index[::-1], top10.values[::-1], color=colors[::-1],
                  edgecolor='#0d1117', height=0.7)
for bar, val in zip(bars3, top10.values[::-1]):
    ax3.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', color='white', fontsize=9)
ax3.set_title("Top 10 Teams by Total Wins", color='white', fontsize=13, fontweight='bold', pad=12)
ax3.set_xlabel("Total Wins", color='#8b949e', fontsize=11)
ax3.tick_params(colors='#8b949e')
for spine in ['top', 'right']:
    ax3.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax3.spines[spine].set_color('#30363d')

plt.suptitle("International Football Results Analysis (1872–2024)",
             color='white', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary of Findings

| Finding | Result |
|---|---|
| Total matches | 2,776 |
| Date range | 30 Nov 1872 → 30 Jun 2024 |
| Unique countries | 94 |
| Most frequent home team | Australia & Costa Rica (40 each) |
| Average goals per match | 2.32 |
| Highest scoring match | Czech Republic 4–4 Russia (8 goals) |
| Most goals scored | At home (3,591 vs 2,855) |
| Most common scoreline | 2 total goals |
| Home win rate | 42.9% |
| Away win rate | 28.2% |
| Draw rate | 28.9% |
| Home advantage confirmed? | ✅ Yes (14.7pp gap) |
| Most wins all-time | Namibia (33) |